In [ ]:
import sys
import os
import json
import pandas as pd


if 'google.colab' in sys.modules: 
    if not os.path.exists('/content/nlp'):
        !git clone https://github.com/jaYulichka46/nlp.git
    %cd /content/nlp
    sys.path.append('/content/nlp')
    data_dir = 'data'
else:
    sys.path.append(os.path.abspath('..'))
    data_dir = '../data'

from src.ie_rules import extract_all


In [ ]:
# 2. Завантаження даних та підрахунок Precision
gold_path = os.path.join(data_dir, "sample", "lab4_gold_ie.jsonl")

# Змінні для підрахунку (True Positives, False Positives)
metrics = {
    "SALARY": {"TP": 0, "FP": 0, "FN": 0},
    "EXPERIENCE_YEARS": {"TP": 0, "FP": 0, "FN": 0},
    "ENGLISH_LEVEL": {"TP": 0, "FP": 0, "FN": 0}
}

false_positives = [] # Для Error Analysis

with open(gold_path, 'r', encoding='utf-8') as f:
    for line in f:
        case = json.loads(line)
        text = case["text"]
        gold_entities = case["gold"]
        
        # Витягуємо сутності через наш скрипт
        predicted_entities = extract_all(text)
        
        # Перетворюємо predicted у зручний формат для порівняння:
        # Для зарплати порівнюємо мінімальне значення
        pred_flat = []
        for p in predicted_entities:
            val = p["value"]["min"] if p["field_type"] == "SALARY" else p["value"]
            pred_flat.append({"type": p["field_type"], "value": val, "raw_dict": p})
            
        gold_flat = [{"type": g["type"], "value": g["value"]} for g in gold_entities]
        
        # Підрахунок метрик
        for pred in pred_flat:
            # Шукаємо збіг у Gold
            match = next((g for g in gold_flat if g["type"] == pred["type"] and g["value"] == pred["value"]), None)
            
            if match:
                metrics[pred["type"]]["TP"] += 1
                gold_flat.remove(match) # Видаляємо, щоб не рахувати двічі
            else:
                metrics[pred["type"]]["FP"] += 1
                false_positives.append({
                    "text": text,
                    "field_type": pred["type"],
                    "extracted_value": pred["value"],
                    "reason": "False Positive (Хибне спрацювання)"
                })

# 3. Розрахунок Precision: TP / (TP + FP)
print("📊 РЕЗУЛЬТАТИ ОЦІНКИ (PRECISION)")
print("="*40)
for field, counts in metrics.items():
    tp, fp = counts["TP"], counts["FP"]
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    print(f"{field.ljust(18)}: Precision = {precision:.2f} (TP:{tp}, FP:{fp})")
print("="*40)

In [ ]:
# 5. Генерація звіту для GitHub
summary_path = "docs/audit_summary_lab4.md" if 'google.colab' in sys.modules else "../docs/audit_summary_lab4.md"

precision_salary = metrics["SALARY"]["TP"] / max((metrics["SALARY"]["TP"] + metrics["SALARY"]["FP"]), 1)
precision_exp = metrics["EXPERIENCE_YEARS"]["TP"] / max((metrics["EXPERIENCE_YEARS"]["TP"] + metrics["EXPERIENCE_YEARS"]["FP"]), 1)
precision_eng = metrics["ENGLISH_LEVEL"]["TP"] / max((metrics["ENGLISH_LEVEL"]["TP"] + metrics["ENGLISH_LEVEL"]["FP"]), 1)

markdown_content = f"""# Audit Summary Lab 4: Rule-based Information Extraction

## 1. Загальна інформація
* **Домен:** Вакансії DOU.ua (Data Engineering & ML)
* **Метод:** Rule-based (Regex + Dictionaries)
* **Поля для екстракції:** `SALARY`, `EXPERIENCE_YEARS`, `ENGLISH_LEVEL`

## 2. Метрики якості (Gold Subset)
Фокус цієї лабораторної — **Precision-first**. Ми оцінювали здатність правил не витягувати "сміття".

| Field Type | Precision | True Positives | False Positives |
| :--- | :---: | :---: | :---: |
| **SALARY** | {precision_salary:.2f} | {metrics['SALARY']['TP']} | {metrics['SALARY']['FP']} |
| **EXPERIENCE_YEARS** | {precision_exp:.2f} | {metrics['EXPERIENCE_YEARS']['TP']} | {metrics['EXPERIENCE_YEARS']['FP']} |
| **ENGLISH_LEVEL** | {precision_eng:.2f} | {metrics['ENGLISH_LEVEL']['TP']} | {metrics['ENGLISH_LEVEL']['FP']} |

## 3. Error Analysis та Edge Cases
* **Проблема з віком:** Алгоритм успішно ігнорує "від 18 років", оскільки ми встановили верхню межу досвіду у 15 років.
* **B2B vs B2:** Використання меж слів (`\b`) дозволило успішно відрізнити вимогу до англійської (B2) від досвіду в B2B продажах.
* **Тривалість проєкту:** Фрази типу "проєкт на 2 роки" можуть викликати False Positives у майбутньому, якщо не додати негативні lookahead патерни.
"""

os.makedirs(os.path.dirname(summary_path), exist_ok=True)
with open(summary_path, "w", encoding="utf-8") as f:
    f.write(markdown_content)

print(f"✅ Звіт успішно згенеровано: {summary_path}")